# Encoder: kandidat graph untuk percepatan iPhone

Tujuan: **memperkecil model dan mempercepat inferensi sambil mempertahankan kualitas label**. Notebook ini tidak menjalankan training atau eksperimen adapter baru.

Kita mengekspor kandidat graph dengan **bobot encoder asli yang sama**, attention dengan tensor rank lebih rendah, dan gerbang convolution yang dinyatakan secara eksplisit. Ini hipotesis perbaikan kompatibilitas GPU; belum bukti bahwa crash MPS sudah selesai. Model aplikasi, boundary Apple, label dan routing CPU aplikasi tetap utuh.

Ekspor memakai shape konkret **750 dan 3000 frame**, sesuai 30/120 detik. Setiap profile memeriksa implementasi asli, graph kandidat, trace, dan Core ML FP32 pada input audio nyata. Jika tidak setara, proses berhenti sebelum kompresi. Selanjutnya bobot dikompresi palette6 dan perubahannya dilaporkan. Selisih akibat kompresi belum dinyatakan aman untuk label.

Mulai dengan **MusicFM**. Setelah hasil iPhone ditinjau, kita dapat menjalankan MuQ melalui fungsi yang sama. Tidak perlu mengekspor atau mengintegrasikan keduanya sekaligus.


In [ ]:
from pathlib import Path
from datetime import datetime
import json, subprocess

ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/encoder_acceleration.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=ROOT.parent/'Section-Annotations'
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'  # Pertahankan executable venv; jangan resolve symlink Python.
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'
RUN=ROOT/'outputs.local'/('encoder-acceleration-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
subprocess.run([str(RUN_PYTHON),'-c','import sys,numpy,torch,coremltools; print("Python:",sys.executable); print("Torch:",torch.__version__); print("CoreMLTools:",coremltools.__version__)'],check=True)

# Laporan sumber model asli dari ekspor sebelumnya, bukan hasil adapter atau model baru.
source_reports={}
for role,filename,flag in [('musicfm','repair_report.json','exact_window_passed'),('muq','export_report.json','source_window_passed')]:
    candidates=[]
    for path in (SONGSTRUCT_ROOT/'outputs.local').glob('*/'+filename):
        r=json.loads(path.read_text())
        if r.get(flag) and r.get('compute_precision')=='float32' and not r.get('compression_scheme'):
            if role=='musicfm' and r.get('musicfm_weights_sha256'):candidates.append(path)
            if role=='muq' and r.get('source_kind')=='original_training_snapshot' and r.get('muq_snapshot_digest'):candidates.append(path)
    if candidates:source_reports[role]=max(candidates,key=lambda p:p.stat().st_mtime)
assert 'musicfm' in source_reports, 'Original MusicFM repair_report.json belum ditemukan; periksa lokasi outputs.local di Section-Annotations'
print('Laporan sumber:',source_reports)
print('Output eksperimen:',RUN)

# Jika laporan sumber berada di folder lain, isi source_reports['musicfm'] / ['muq'] dengan Path yang tepat.
def export_encoder(role):
    assert role in source_reports, f'Laporan sumber {role} belum ditemukan'
    output=RUN/role
    if (output/'Benchmark.xcodeproj/project.pbxproj').is_file():
        print('Project sudah tersedia:',output/'Benchmark.xcodeproj')
        return output
    if output.exists():
        raise RuntimeError(f'Ekspor sebelumnya belum selesai: {output}. Periksa log; gunakan RUN baru untuk percobaan ulang tanpa menimpa bukti.')
    command=[str(RUN_PYTHON),'-u',str(ROOT/'scripts/encoder_acceleration.py'),
        '--songstruct-root',str(SONGSTRUCT_ROOT),'--songmap-root',str(ROOT),
        '--encoder',role,'--source-report',str(source_reports[role]),'--scheme','palette6','--output-dir',str(output)]
    log=RUN/(role+'.export.log');log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'Ekspor berhenti ({status}); lihat {log}. Jangan lanjut integrasi aplikasi.')
    proof=json.loads((output/'export-report.json').read_text())
    assert proof['same_weight_rewrite_passed'] and not proof['ready_for_app_integration']
    for item in proof['checks']:
        print('Frames:',item['frames'],'| rewrite:',item['rewrite_vs_original'],'| Core ML FP32:',item['coreml_fp32_vs_original'])
        print('Perubahan setelah kompresi:',item['compression_vs_fp32'])
    print('Kirim laporan ekspor:',output/'export-report.json')
    return output


In [ ]:
# Ekspor MusicFM saja terlebih dahulu. Model besar dikonversi bergantian per profile.
MUSICFM=export_encoder('musicfm')
subprocess.run(['open',str(MUSICFM/'Benchmark.xcodeproj')],check=True)


## Pengukuran iPhone

Di project **benchmark terpisah** yang terbuka, pilih Signing Team dan iPhone fisik. Jalankan aplikasi; jangan ubah routing di project SongMap utama.

1. Profile `musicfm_750`, mode `cpuOnly`: jalankan dan **Share JSON report**.
2. Tutup aplikasi sepenuhnya, buka lagi. Profile yang sama, mode `cpuAndGPU`: jalankan dan bagikan JSON.
3. Ulangi dua langkah itu untuk `musicfm_3000`.

Setiap launch hanya mengizinkan satu pengukuran agar model/routing sebelumnya tidak bercampur. Setiap pengukuran merekam load, prediksi pertama, tiga prediksi berikutnya, kondisi thermal, snapshot memori dan perbedaan output terhadap kandidat terkompresi yang sama pada CPU Mac. Mode `cpuAndNeuralEngine` tersedia untuk pengujian terpisah; memilih mode tidak membuktikan operasi benar-benar dijalankan di ANE.

Jika aplikasi probe crash, buka lagi dan bagikan **last-stage JSON** melalui tombol Share sebelum memulai run berikutnya, beserta log error Xcode. Progress ditulis sebelum load/prediksi agar titik terakhir tetap tersedia meskipun proses dihentikan. Kegagalan GPU fatal tidak bisa ditangkap melalui Swift `catch`.

Kirim empat JSON benchmark dan `export-report.json`. Kecepatan dihitung per profile dan versi graph yang sama; hasil warm saja tidak cukup untuk menyatakan waktu analisis satu lagu sudah lebih cepat.


In [ ]:
# Jalankan SETELAH hasil MusicFM ditinjau, jika kita lanjut menguji MuQ.
# MUQ=export_encoder('muq')
# subprocess.run(['open',str(MUQ/'Benchmark.xcodeproj')],check=True)


Project benchmark memakai file statis terpisah untuk mengisolasi shape; bobot terduplikasi sementara. Ukuran total project benchmark bukan keputusan packaging produksi. Setelah jalur cepat terbukti stabil, packaging bobot bersama dan ukuran paket aplikasi harus dievaluasi tersendiri.

Benchmark ini hanya encoder dengan input nyata tetap, belum frontend, seluruh lagu, head, label manual, boundary atau total waktu aplikasi. Kandidat yang cepat masih harus melewati validation label dengan pooling/boundary tetap dan uji beberapa lagu/cold launch di iPhone sebelum mengganti jalur aplikasi. Tidak ada klaim percepatan, pemakaian ANE/GPU, atau perbaikan crash sampai ada hasil perangkat.
